# Training Notebook

### Objectives:
- Create a random forest regressor based model
- Use scikit-learn
- Explore the data to find the most important deciders of weather the flight is delayed
- Graph these explorations
- Split the datasets
- Get decent accuracy with the validation dataset

In [12]:
# Scikit imports
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error


In [13]:
# Imports
import duckdb as ddb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from memory_profiler import memory_usage
from time import perf_counter
import numpy as np
from scipy.stats import randint

# Constants

# run the proj in backend. This is because locating via Path(__file__) does not work with notebooks
DUCKDB_PATH = Path().cwd().resolve().parents[2]/"data/duck_database.duckdb"

In [68]:
# Getting data_df
con = ddb.connect(DUCKDB_PATH)
data_df = con.sql("""
    SELECT * FROM model_dataset LIMIT 50000
""").df()
con.close()

data_df = data_df.dropna(axis=0)

In [69]:
# Create X's and y's
x_numeric_features = ['pred_dep_time', 'pred_arr_time', 'pred_elapsed_time',
       'fl_distance', 'origin_weather_code',
       'origin_temperature_2m_max', 'origin_temperature_2m_min',
       'origin_apparent_temperature_max', 'origin_apparent_temperature_min',
       'origin_precipitation_sum', 'origin_rain_sum', 'origin_showers_sum',
       'origin_snowfall_sum', 'origin_cloud_cover_mean',
       'origin_wind_speed_10m_max', 'origin_wind_gusts_10m_max',
       'origin_wind_direction_10m_dominant', 'origin_pressure_msl_mean',
       'dest_weather_code', 'dest_temperature_2m_max',
       'dest_temperature_2m_min', 'dest_apparent_temperature_max',
       'dest_apparent_temperature_min', 'dest_precipitation_sum',
       'dest_rain_sum', 'dest_showers_sum', 'dest_snowfall_sum',
       'dest_cloud_cover_mean', 'dest_wind_speed_10m_max',
       'dest_wind_gusts_10m_max', 'dest_wind_direction_10m_dominant',
       'dest_pressure_msl_mean']
x_categorical_features = ['flight_date', 'origin', 'dest']
x_features = x_numeric_features + x_categorical_features

X = data_df[x_features]
y = data_df["delay"]

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

### Data exploration
- With notes on matplotlab (not great with it)

In [70]:
# Correlations

numeric_features = data_df[x_numeric_features + ["delay"]].dropna(axis=0).select_dtypes(include="number") # this essentially creates the df necessary for the corr table. drops all that are null and ensures all are nums
delay_correlations = (
    numeric_features
    .corr(numeric_only=True)["delay"]
    .drop("delay")
    .dropna()
    .sort_values(key=lambda values: values.abs()) # makes (for ex) -.5 a greater value than 0.2 (good for graph)
)

fig, ax = plt.subplots(figsize=(11, max(8, 0.34 * len(delay_correlations)))) # creates the plot and the spaces for each row/ the plot
colors = ["#b45309" if value < 0 else "#0f766e" for value in delay_correlations]

ax.barh(delay_correlations.index, delay_correlations.values, color=colors) # draws a horizontal bar chart. Preyy self explanotiry if you look at vars
ax.axvline(0, color="#222222", linewidth=0.8) # adds the middle line to show the start for all charts
ax.set_title("Pre-exploration Pearson correlation with delay") # title of the 'set' (table)
ax.set_xlabel("Correlation with delay") # title of the x
ax.set_ylabel("Numeric feature") # title of the y
ax.grid(axis="x", alpha=0.25) # adds the other lines allong the points so things are visable (poor epxlination but i mean the mildly seethorugh lines on the table)

fig.tight_layout() # auto does spacing so it looks sweet
# plt.show() # showing it in the notbook
fig.savefig("figures/correlation_chart.png", dpi=300, bbox_inches="tight")
plt.close(fig)

In [71]:
# Distribution
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(data_df["delay"].dropna(), bins=range(-60, 301, 10), color="slategray", edgecolor="white")

ax.set_title("Distribution of flight delays")
ax.set_xlabel("Delay (mins)")
ax.set_ylabel("Number of flights")

ax.set_xlim(-10, 151)

fig.tight_layout()
fig.savefig("figures/delay_distribution.png", dpi=300, bbox_inches="tight")
plt.close(fig)

In [72]:
# Missing values

missing = data_df[x_features + ["delay"]].isna().mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(missing.index, missing.values)

ax.set_title("Missing values in data")
ax.set_xlabel("Fraction missing")
ax.set_ylabel("Feature")

fig.tight_layout()
fig.savefig("figures/missing_values.png", dpi=300, bbox_inches="tight")
plt.close(fig)

# No missing values, good :)

### Training
Methods for accuracy improvments:
- Duplicate accurate indicators (shown in correlation img)
- Combine features, for example: snow + wind
- Train many models

In [ ]:
param_grid = {
    'classifier__n_estimators':[100, 200, 300, 400, 500, 600,700, 800, 900],
    'classifier__max_depth':[None, 5, 10, 20, 30, 40, 50],
    'classifier__min_samples_split':[2, 5, 8, 10, 15, 20],
    'classifier__min_samples_leaf':[1, 2, 4, 8, 12],
    'classifier__max_features': ["sqrt", "log2", 0.3, 0.5, 0.7, 1.0],
    'classifier__bootstrap': [True, False],
    'classifier__max_samples': [0.5, 0.7, 0.9]
}

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), x_categorical_features),
        ("numeric", "passthrough", x_numeric_features),
    ]
)
pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("classifier", RandomForestRegressor(n_jobs=1, random_state=1)),
    ]
)
grid_search = RandomizedSearchCV(pipeline, param_grid, cv=2, n_jobs=-1)
grid_search.fit(X_train, y_train)

/Users/magnusnewton/Desktop/code/storm_proj/backend/.venv/lib/python3.13/site-packages/sklearn/model_selection/_search.py:326: UserWarning: The total space of parameters 3 is smaller than n_iter=10. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...om_state=1))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__bootstrap': [True], 'classifier__max_depth': [18], 'classifier__max_features': ['log2'], 'classifier__max_samples': [0.5], ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",2
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example<sphx_glr_auto_examp

In [78]:
preds = grid_search.predict(X_test)

mea = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)
rmse = root_mean_squared_error(y_test, preds)

print("MEA: " + str(mea))
print("R2: " + str(r2))
print("RMSE: " + str(rmse))

MEA: 14.687286650251405
R2: 0.012751840698456918
RMSE: 35.27338195447001


In [79]:
grid_search.best_params_

{'classifier__n_estimators': 420,
 'classifier__min_samples_split': 18,
 'classifier__min_samples_leaf': 11,
 'classifier__max_samples': 0.5,
 'classifier__max_features': 'log2',
 'classifier__max_depth': 18,
 'classifier__bootstrap': True}

In [76]:
grid_search.best_estimator_

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](35,)","['pred_dep_time','pred_arr_time','pred_elapsed_time',...,'flight_date', 'origin','dest']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,35
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed